# This file is for Experimenting and testing logic

In [1]:
print("Hello")

Hello


In [121]:
import os
import json
import pandas as pd
import traceback
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

In [122]:
load_dotenv(override=True)

LLM_MODEL = os.getenv("OLLAMA_MODEL")

print(f"LLM_MODEL: {LLM_MODEL}")

LLM_MODEL: gemma4


In [123]:
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0.3,
    streaming=True,
    verbose=True,
)

In [124]:
respone = llm.invoke("What is GenAI, and how it is different from traditional AI?")
print(f"Response: {respone.content}")

Response: This is one of the most important distinctions in modern technology. While both Generative AI and traditional AI fall under the umbrella of "Artificial Intelligence," they operate on fundamentally different principles, have different goals, and produce vastly different types of outputs.

Here is a detailed breakdown of what GenAI is, followed by a clear comparison to traditional AI.

---

## 🧠 Part 1: What is Generative AI (GenAI)?

**Generative AI** refers to artificial intelligence models that are capable of **creating new, original content**—content that did not exist before the model was run. Instead of merely analyzing data and telling you what something *is*, it synthesizes data and tells you what something *could be*.

### How It Works
GenAI models are trained on massive datasets (trillions of words, millions of images, etc.). During training, they don't just memorize the data; they learn the underlying **patterns, relationships, and structure** that govern that data. 

In [125]:
from langchain_classic.llms import Ollama
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2

In [126]:
RESPONSE_JSON_STRUCTURE = {
    '1': {
        'mcq': 'multiple choice question',
        'options': {
            'a': 'choice here',
            'b': 'choice here',
            'c': 'choice here',
            'd': 'choice here'
        },
        'correct_answer': 'correct choice here',
    },
    '2': {
        'mcq': 'multiple choice question',
        'options': {
            'a': 'choice here',
            'b': 'choice here',
            'c': 'choice here',
            'd': 'choice here'
        },
        'correct_answer': 'correct choice here',
    }
}

In [127]:
TEMPLATE_QUIZ_GENERATION_PROMPT = """
You are an expert in {subject} and have been asked to create a multiple-choice quiz based on the following text:
{text} 
create a quiz with {number} questions, each with 4 options. The quiz should be in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming to the text provided. 
The output should be in JSON format as per the following structure: {response_json}
Ensure that the questions are clear, concise, and relevant to the text provided.
And ensure to make {number} questions in the quiz.
"""

In [128]:
quiz_generation_prompt_template = PromptTemplate(
    input_variables=['text', 'number', 'subject', 'tone', 'response_json'],
    template=TEMPLATE_QUIZ_GENERATION_PROMPT
)

In [129]:
quiz_chain = LLMChain(
    llm=llm,
    prompt=quiz_generation_prompt_template,
    output_key='quiz',
    verbose=True
)

In [130]:
TEMPLATE2 = """
You are an expert english grammarian and writer. Give me a multiple-choice quiz for {subject} students.
You need to evaluate the complexity of the questions and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. The quiz should be in {tone} tone.
If the quiz is not good enough, you need to give a complete analysis of the quiz and suggest improvements.
Update the quiz based on your analysis and suggestions and change the tone such that it is more suitable for the students.
Quiz_MCQs: 
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [131]:
chain2_prompt_template = PromptTemplate(
    input_variables=['quiz', 'subject', 'tone'],
    template=TEMPLATE2
)

In [132]:
quiz_evaluation_chain = LLMChain(
    llm=llm,
    prompt=chain2_prompt_template,
    output_key='review',
    verbose=True
)

In [133]:
generate_evaluation_chain = SequentialChain(
    chains=[quiz_chain, quiz_evaluation_chain],
    input_variables=['text', 'number', 'subject', 'tone', 'response_json'],
    output_variables=['quiz', 'review'],
    verbose=True
)

In [134]:
file_path = "/Users/mdirfan/dev/python/GenAI-MCQ/data.txt"

In [135]:
with open(file_path, 'r') as f:
    TEXT = f.read()

In [136]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied human cognitive systems contributed to the m

In [137]:
json.dumps(RESPONSE_JSON_STRUCTURE)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct_answer": "correct choice here"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct_answer": "correct choice here"}}'

In [138]:
with get_openai_callback() as cb:
    generated_quiz = generate_evaluation_chain(
        {
            "text": TEXT,
            "number": 10,
            "subject": "machine learning",
            "tone": "simple",
            "response_json": json.dumps(RESPONSE_JSON_STRUCTURE)
        }
    )
    print(f"Total Tokens Used: {cb.total_tokens}")
    print(f"Prompt Tokens Used: {cb.prompt_tokens}")
    print(f"Completion Tokens Used: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost:.6f}")



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert in machine learning and have been asked to create a multiple-choice quiz based on the following text:
The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwor

In [139]:
generated_quiz

{'text': 'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]\n\nThe earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied human cognitive systems contribu

In [140]:
quiz = generated_quiz.get('quiz')

In [141]:
review = generated_quiz.get('review')

In [142]:
print(review)

As an expert English Grammarian and writer, I have analyzed your quiz.

The grammar and syntax of the questions are technically correct. However, the language is highly academic and dense, which can make the quiz feel intimidating or overly formal for students who might be learning the material for the first time. The tone needs to be adjusted to be more encouraging and conversational while maintaining its expert rigor.

***

### Complexity Analysis
The complexity is **High**. It requires deep historical knowledge (dates, names) combined with specific technical definitions (GANs, Turing Test). This makes it suitable for advanced graduate students or those who have already covered the foundational history of ML in depth.

***

### Suggested Improvements and Updated Quiz

**Analysis Summary:**
The content is excellent—it covers crucial historical milestones (Turing, McCulloch/Pitts, Samuel, AlphaGo) that are essential for understanding ML's evolution. The primary improvement needed is a 

In [143]:
print(quiz)

```json
{
  "1": {
    "mcq": "Who is credited with coining the term 'machine learning' in 1959?",
    "options": {
      "a": "Donald Hebb",
      "b": "Alan Turing",
      "c": "Arthur Samuel",
      "d": "Nils Nilsson"
    },
    "correct_answer": "c"
  },
  "2": {
    "mcq": "What synonym was used for machine learning during its early period, according to the text?",
    "options": {
      "a": "Artificial Intelligence",
      "b": "Self-teaching computers",
      "c": "Cognitive Computing",
      "d": "Pattern Recognition Systems"
    },
    "correct_answer": "b"
  },
  "3": {
    "mcq": "Which psychologist published 'The Organization of Behavior' in 1949, laying groundwork for neural theories?",
    "options": {
      "a": "Walter Pitts",
      "b": "Warren McCulloch",
      "c": "Donald Hebb",
      "d": "Tom M. Mitchell"
    },
    "correct_answer": "c"
  },
  "4": {
    "mcq": "What early experimental machine, developed by Raytheon Company in the 1960s, analyzed signals like s

In [144]:
quiz_str = quiz.replace("```json", "").replace("```", "").strip()

In [145]:
quiz_str

'{\n  "1": {\n    "mcq": "Who is credited with coining the term \'machine learning\' in 1959?",\n    "options": {\n      "a": "Donald Hebb",\n      "b": "Alan Turing",\n      "c": "Arthur Samuel",\n      "d": "Nils Nilsson"\n    },\n    "correct_answer": "c"\n  },\n  "2": {\n    "mcq": "What synonym was used for machine learning during its early period, according to the text?",\n    "options": {\n      "a": "Artificial Intelligence",\n      "b": "Self-teaching computers",\n      "c": "Cognitive Computing",\n      "d": "Pattern Recognition Systems"\n    },\n    "correct_answer": "b"\n  },\n  "3": {\n    "mcq": "Which psychologist published \'The Organization of Behavior\' in 1949, laying groundwork for neural theories?",\n    "options": {\n      "a": "Walter Pitts",\n      "b": "Warren McCulloch",\n      "c": "Donald Hebb",\n      "d": "Tom M. Mitchell"\n    },\n    "correct_answer": "c"\n  },\n  "4": {\n    "mcq": "What early experimental machine, developed by Raytheon Company in the 1

In [146]:
json_output = json.dumps(quiz_str, indent=4)

In [147]:
quiz = json.loads(quiz_str)

In [148]:
print(type(quiz))

<class 'dict'>


In [149]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value.get('mcq')
    options = " | ".join(
        [
            f"{opt_key}: {opt_value}" for opt_key, opt_value in value.get('options', {}).items()
        ]
    ),
    correct_answer = value.get('correct_answer')
    quiz_table_data.append([key, mcq, options, correct_answer])

In [150]:
quiz_table_data

[['1',
  "Who is credited with coining the term 'machine learning' in 1959?",
  ('a: Donald Hebb | b: Alan Turing | c: Arthur Samuel | d: Nils Nilsson',),
  'c'],
 ['2',
  'What synonym was used for machine learning during its early period, according to the text?',
  ('a: Artificial Intelligence | b: Self-teaching computers | c: Cognitive Computing | d: Pattern Recognition Systems',),
  'b'],
 ['3',
  "Which psychologist published 'The Organization of Behavior' in 1949, laying groundwork for neural theories?",
  ('a: Walter Pitts | b: Warren McCulloch | c: Donald Hebb | d: Tom M. Mitchell',),
  'c'],
 ['4',
  'What early experimental machine, developed by Raytheon Company in the 1960s, analyzed signals like sonar and speech?',
  ('a: AlphaGo | b: The Checker Program | c: Cybertron | d: GAN',),
  'c'],
 ['5',
  "According to Tom M. Mitchell's definition, what are the three components used to describe if a program is 'learning'?",
  ('a: Data, Algorithms, and Tasks | b: Experience (E), T

In [151]:
quiz = pd.DataFrame(quiz_table_data)

In [152]:
quiz.to_csv("machine_learning_quiz.csv", index=False)